# 03 — Explorer Quickstart

The **Explorer** module answers a critical question before you invest time in anomaly detection:

> **Does my data actually contain enough signal to detect anomalies?**

It runs a battery of automated checks on your dataset and tells you whether to proceed, collect more data, or engineer better features.

### What it checks

| Check | What it measures | Why it matters |
|-------|-----------------|----------------|
| `min_entries` | Number of non-null rows | Too few samples → unreliable models |
| `min_non_null_pct` | Completeness (% non-null) | Sparse data → biased aggregations |
| `min_variance` | Signal variability | Near-constant columns carry no information |
| `anomaly_pct` | IQR outlier percentage | No outliers → nothing to detect |
| `correlation` | Feature-label correlation | Weak correlation → features aren't predictive |

### Key classes

- `SignalDiagnostics` — runs all checks programmatically
- `Thresholds` — configurable pass/fail criteria (`.default()`, `.strict()`, `.relaxed()`)
- `QualityReport` — structured results with `.interpret()` for human-readable recommendations
- `detect_anomalies()` — standalone IQR-based outlier detection
- `detect_drift()` — statistical distribution shift detection

In [1]:
import numpy as np
import pandas as pd

from sentinel.explorer import (
    SignalDiagnostics,
    Thresholds,
    detect_anomalies,
    detect_drift,
)

---
## 1. Generate Synthetic Data

We create a dataset simulating server metrics with:
- 500 normal samples (cpu ~50%, memory ~2048 MB)
- 30 anomalous samples (cpu ~95%, memory ~7500 MB)
- A binary `label` column (0 = normal, 1 = anomaly)

In [2]:
rng = np.random.RandomState(42)
n_normal, n_anomaly = 500, 30

df = pd.DataFrame({
    'cpu': np.concatenate([rng.normal(50, 8, n_normal), rng.normal(95, 3, n_anomaly)]),
    'memory_mb': np.concatenate([rng.normal(2048, 200, n_normal), rng.normal(7500, 150, n_anomaly)]),
    'label': [0] * n_normal + [1] * n_anomaly,
})

print(f"Shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.describe()

Shape: (530, 3)
Label distribution:
label
0    500
1     30
Name: count, dtype: int64


,cpu,memory_mb,label
count,530.000000,530.000000,530.000000
mean,52.554850,2371.537056,0.056604
std,12.762435,1277.357319,0.231302
min,24.069861,1508.622671,0.000000
25%,44.797252,1940.274287,0.000000
50%,50.612691,2079.675397,0.000000
75%,56.286153,2220.473065,0.000000
max,100.728250,7752.589154,1.000000


---
## 2. Quick IQR Anomaly Detection

`detect_anomalies(df, column)` uses the Interquartile Range (IQR) method:
- Computes Q1 (25th percentile) and Q3 (75th percentile)
- IQR = Q3 - Q1
- Flags any value outside `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]` as an anomaly

This is a fast, non-parametric baseline — no training required.

In [3]:
anomalies_cpu = detect_anomalies(df, 'cpu')
anomalies_mem = detect_anomalies(df, 'memory_mb')

print(f"IQR anomalies in 'cpu':       {len(anomalies_cpu):>3} ({len(anomalies_cpu)/len(df)*100:.1f}%)")
print(f"IQR anomalies in 'memory_mb': {len(anomalies_mem):>3} ({len(anomalies_mem)/len(df)*100:.1f}%)")
print(f"\nSample anomalous rows (cpu):")
anomalies_cpu.head()

IQR anomalies in 'cpu':        33 (6.2%)
IQR anomalies in 'memory_mb':  32 (6.0%)

Sample anomalous rows (cpu):


,cpu,memory_mb,label
209,80.821852,2250.874013,0
262,24.069861,2102.315767,0
478,74.631046,2257.910543,0
500,97.778533,7461.393519,1
501,100.728250,7249.712389,1


The IQR method detected the injected anomalies because they sit far outside the normal distribution. This is a good sign — it means the anomalies are statistically distinguishable from normal data.

---
## 3. SignalDiagnostics — Statistical Summary

The `summary()` method provides per-column statistics including variance, quartiles, and IQR anomaly counts. This helps you understand the data distribution before running checks.

In [4]:
diag = SignalDiagnostics(df, columns=['cpu', 'memory_mb'], label_column='label')

summary = diag.summary()
pd.DataFrame(summary).T

,count,null_pct,mean,std,variance,q25,q50,q75,iqr_anomaly_count,iqr_anomaly_pct
cpu,530.0,0.0,52.554850,12.762435,1.628797e+02,44.797252,50.612691,56.286153,33.0,6.23
memory_mb,530.0,0.0,2371.537056,1277.357319,1.631642e+06,1940.274287,2079.675397,2220.473065,32.0,6.04


### How to read the summary

| Metric | What to look for |
|--------|------------------|
| `null_pct` | Should be low. High values mean missing data that could bias results. |
| `variance` | Should be non-trivial. Near-zero variance means the column is essentially constant. |
| `iqr_anomaly_pct` | Should be > 0. If there are zero IQR outliers, the data may be too uniform for anomaly detection. |

---
## 4. Quality Report with Thresholds

The `quality_report()` method runs all checks against configurable thresholds and returns a `QualityReport` with pass/fail results.

### Threshold presets

| Preset | `min_entries` | `min_non_null_pct` | `min_variance` | `anomaly_pct` | `correlation` | Use case |
|--------|--------------|-------------------|---------------|--------------|--------------|----------|
| `default()` | 1,000 | 95% | 0.01 | 5% | 0.3 | Production-ready data |
| `strict()` | 25,000 | 99% | 500 | 5% | 0.4 | High-confidence analysis |
| `relaxed()` | 100 | 80% | 0.001 | 1% | 0.1 | Exploratory / small samples |

In [5]:
for name, thresholds in [('default', Thresholds.default()),
                          ('strict', Thresholds.strict()),
                          ('relaxed', Thresholds.relaxed())]:
    report = diag.quality_report(thresholds)
    n_failed = len(report.failed_checks)
    print(f"{name:>8}: {report}")
    if n_failed > 0:
        for c in report.failed_checks:
            print(f"          {c}")

 default: QualityReport(FAILED, 10 checks, 2 failed)
          [FAIL] min_entries | cpu: 530.0000 (threshold=1000)
          [FAIL] min_entries | memory_mb: 530.0000 (threshold=1000)
  strict: QualityReport(FAILED, 10 checks, 3 failed)
          [FAIL] min_entries | cpu: 530.0000 (threshold=25000)
          [FAIL] min_variance | cpu: 162.8797 (threshold=500.0)
          [FAIL] min_entries | memory_mb: 530.0000 (threshold=25000)
 relaxed: QualityReport(PASSED, 10 checks, 0 failed)


### Interpreting the results

With our 530-row dataset:
- `default` fails on `min_entries` (needs 1000, we have 530) — this is expected for a small sample
- `strict` also fails on `min_variance` for cpu — the strict threshold (500) is higher than cpu's variance (~163)
- `relaxed` passes everything — the data has enough signal for exploratory analysis

The key insight: **failing `min_entries` alone doesn't mean the data is bad** — it means you should be cautious about generalizing results from a small sample.

---
## 5. Automated Interpretation

The `interpret()` method on `QualityReport` generates a human-readable diagnosis with specific recommendations based on which checks failed.

In [6]:
report = diag.quality_report(Thresholds.default())
print(report.interpret())

INSUFFICIENT SIGNAL — 2/10 checks failed (score: 80%).

  [min_entries] Failed for: cpu, memory_mb
    → Not enough data points. The dataset has fewer rows than the required minimum. Collect more data or use Thresholds.relaxed() for exploratory analysis.

Recommendation: The only issue is data volume. If this is a sample, try with the full dataset. Otherwise, use Thresholds.relaxed() to proceed with exploratory analysis.


In [7]:
# Compare with strict thresholds
report_strict = diag.quality_report(Thresholds.strict())
print(report_strict.interpret())

INSUFFICIENT SIGNAL — 3/10 checks failed (score: 70%).

  [min_entries] Failed for: cpu, memory_mb
    → Not enough data points. The dataset has fewer rows than the required minimum. Collect more data or use Thresholds.relaxed() for exploratory analysis.

  [min_variance] Failed for: cpu
    → Near-constant signal. The column has very low variance, meaning there is little variation for a detector to learn from. Check if the column is informative or if the data needs a different time range.

Recommendation: Review whether low-variance columns carry useful information. Drop or transform them before detection.


In [8]:
# And relaxed — should give the green light
report_relaxed = diag.quality_report(Thresholds.relaxed())
print(report_relaxed.interpret())

SIGNAL DETECTED — All 10 checks passed (score: 100%).
The data has sufficient volume, variance, and anomaly presence to proceed with anomaly detection.


### Using the score

The `score` property gives you a 0.0–1.0 value representing the fraction of checks that passed. You can use this programmatically to gate your pipeline:

In [9]:
report = diag.quality_report(Thresholds.relaxed())

print(f"Score: {report.score:.0%}")

if report.score >= 0.8:
    print("→ Sufficient signal. Proceeding with anomaly detection.")
elif report.score >= 0.5:
    print("→ Partial signal. Review failed checks before proceeding.")
else:
    print("→ Weak signal. Collect more data or engineer better features.")

Score: 100%
→ Sufficient signal. Proceeding with anomaly detection.


---
## 6. Quick Report (all-in-one)

`quick_report()` runs all diagnostics in a single call and includes the interpretation.

In [10]:
full = diag.quick_report(thresholds=Thresholds.default())

print("Keys:", list(full.keys()))
print(f"\nQuality: {full['quality_report']}")
print(f"Score: {full['quality_report'].score:.0%}")
print(f"\nAnomaly distribution: {full['anomaly_distribution']}")
print(f"\nPredictive power: {full['predictive_power']}")
print(f"\n{'='*60}")
print(full['interpretation'])

Keys: ['summary', 'quality_report', 'correlation', 'anomaly_distribution', 'predictive_power', 'interpretation']

Quality: QualityReport(FAILED, 10 checks, 2 failed)
Score: 80%

Anomaly distribution: {'cpu': {'method': 'iqr', 'anomaly_count': 33, 'anomaly_pct': 6.23}, 'memory_mb': {'method': 'iqr', 'anomaly_count': 32, 'anomaly_pct': 6.04}}

Predictive power: {'cpu': {'recall': 1.0}, 'memory_mb': {'recall': 1.0}}

INSUFFICIENT SIGNAL — 2/10 checks failed (score: 80%).

  [min_entries] Failed for: cpu, memory_mb
    → Not enough data points. The dataset has fewer rows than the required minimum. Collect more data or use Thresholds.relaxed() for exploratory analysis.

Recommendation: The only issue is data volume. If this is a sample, try with the full dataset. Otherwise, use Thresholds.relaxed() to proceed with exploratory analysis.


### What each section tells you

| Section | Purpose |
|---------|--------|
| `summary` | Per-column statistics (mean, std, variance, quartiles, IQR anomaly %) |
| `quality_report` | Pass/fail checks against thresholds |
| `correlation` | Point-biserial correlation between each feature and the label (if provided) |
| `anomaly_distribution` | How many IQR outliers exist per column |
| `predictive_power` | Single-feature logistic regression recall (can each feature alone predict the label?) |
| `interpretation` | Human-readable diagnosis with recommendations |

---
## 7. Custom Thresholds

You can define your own thresholds for domain-specific requirements.

In [11]:
custom = Thresholds(
    min_entries=100,
    min_non_null_pct=90.0,
    min_variance=10.0,
    correlation_threshold=0.2,
    anomaly_pct_threshold=3.0,
)

report = diag.quality_report(custom)
print(report)
print(f"Score: {report.score:.0%}")
print()
for c in report.checks:
    print(f"  {c}")
print()
print(report.interpret())

QualityReport(PASSED, 10 checks, 0 failed)
Score: 100%

  [PASS] min_entries | cpu: 530.0000 (threshold=100)
  [PASS] min_non_null_pct | cpu: 100.0000 (threshold=90.0)
  [PASS] min_variance | cpu: 162.8797 (threshold=10.0)
  [PASS] anomaly_pct | cpu: 6.2264 (threshold=3.0)
  [PASS] min_entries | memory_mb: 530.0000 (threshold=100)
  [PASS] min_non_null_pct | memory_mb: 100.0000 (threshold=90.0)
  [PASS] min_variance | memory_mb: 1631641.7205 (threshold=10.0)
  [PASS] anomaly_pct | memory_mb: 6.0377 (threshold=3.0)
  [PASS] correlation | cpu: 0.8005 (threshold=0.2)
  [PASS] correlation | memory_mb: 0.9885 (threshold=0.2)

SIGNAL DETECTED — All 10 checks passed (score: 100%).
The data has sufficient volume, variance, and anomaly presence to proceed with anomaly detection.


---
## 8. Correlation Report

The correlation report computes the point-biserial correlation between each numeric feature and the binary label. This tells you how strongly each feature is associated with the anomaly class.

- Values close to ±1.0 → strong association
- Values close to 0 → no linear association
- `p_value` < 0.05 → statistically significant

In [12]:
corr = diag.correlation_report()
for col, info in corr.items():
    sig = '✅ significant' if info.get('p_value', 1) < 0.05 else '⚠️ not significant'
    print(f"{col}: r={info.get('point_biserial', 'N/A')}, p={info.get('p_value', 'N/A'):.2e} ({sig})")

cpu: r=0.8005, p=1.74e-119 (✅ significant)
memory_mb: r=0.9885, p=0.00e+00 (✅ significant)


---
## 9. Predictive Power

Tests whether each feature alone can predict the label using a simple logistic regression. A high recall means the feature is individually informative.

In [13]:
pp = diag.predictive_power()
for col, info in pp.items():
    quality = '🟢 strong' if info['recall'] >= 0.7 else '🟡 moderate' if info['recall'] >= 0.4 else '🔴 weak'
    print(f"{col}: recall={info['recall']} ({quality})")

cpu: recall=1.0 (🟢 strong)
memory_mb: recall=1.0 (🟢 strong)


---
## 10. Without Label Column

Explorer works without a label — it skips label-dependent checks (correlation, predictive power) and focuses on data quality.

In [14]:
diag_no_label = SignalDiagnostics(df, columns=['cpu', 'memory_mb'])
report = diag_no_label.quality_report(Thresholds.relaxed())

print(report)
print(f"Score: {report.score:.0%}")
print()
print(report.interpret())
print(f"\nCorrelation report: {diag_no_label.correlation_report()}")
print(f"Predictive power: {diag_no_label.predictive_power()}")

QualityReport(PASSED, 8 checks, 0 failed)
Score: 100%

SIGNAL DETECTED — All 8 checks passed (score: 100%).
The data has sufficient volume, variance, and anomaly presence to proceed with anomaly detection.

Correlation report: {'cpu': {}, 'memory_mb': {}}
Predictive power: {}


---
## 11. Drift Detection

`detect_drift()` uses the Kolmogorov-Smirnov test to check if the data distribution changes over time. It slides a window across the data and compares each window against a reference window.

- `statistic` close to 0 → distributions are similar
- `p_value` < 0.05 → statistically significant drift detected

This is useful for detecting concept drift — when the data generating process changes and your model may need retraining.

In [15]:
# Create data with a clear distribution shift at row 300
drift_df = pd.DataFrame({
    'metric': np.concatenate([
        rng.normal(10, 1, 300),   # stable period
        rng.normal(12, 1.5, 200), # shifted period
    ])
})

drift_results = detect_drift(drift_df, column='metric', window=100)

print(f"Reference window: rows 0–99")
print(f"Comparison windows (size=100):\n")
for r in drift_results:
    status = '⚠️ DRIFT' if r['drifted'] else '✅ stable'
    print(f"  rows {r['start_idx']:>3}–{r['end_idx']:>3}: "
          f"KS={r['statistic']:.4f}, p={r['p_value']:.6f} → {status}")

Reference window: rows 0–99
Comparison windows (size=100):

  rows 100–200: KS=0.1300, p=0.368188 → ✅ stable
  rows 200–300: KS=0.1300, p=0.368188 → ✅ stable
  rows 300–400: KS=0.6400, p=0.000000 → ⚠️ DRIFT
  rows 400–500: KS=0.6000, p=0.000000 → ⚠️ DRIFT


The first two windows (rows 100–299) come from the same distribution as the reference, so no drift is detected. The last two windows (rows 300–499) come from a shifted distribution (mean 12 vs 10), so drift is flagged.

---
## Decision Flowchart

```
                    ┌─────────────────────┐
                    │  quick_report()      │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │  score >= 80%?       │
                    └──────────┬──────────┘
                       yes     │     no
                  ┌────────────┴────────────┐
                  ▼                         ▼
         Proceed with              Check interpret()
         detection                 for recommendations
                                        │
                              ┌─────────┴─────────┐
                              ▼                   ▼
                     Only min_entries?      Other failures?
                              │                   │
                              ▼                   ▼
                     Use relaxed()         Fix data issues
                     or get more data      before proceeding
```

---
## Summary

| Tool | Purpose |
|------|--------|
| `detect_anomalies(df, col)` | Quick IQR outlier detection |
| `SignalDiagnostics.summary()` | Per-column statistics |
| `SignalDiagnostics.quality_report(thresholds)` | Automated pass/fail checks |
| `QualityReport.interpret()` | Human-readable diagnosis and recommendations |
| `QualityReport.score` | 0.0–1.0 signal quality score for pipeline gating |
| `SignalDiagnostics.quick_report()` | All-in-one: summary + quality + correlation + interpretation |
| `detect_drift(df, col, window)` | Distribution shift detection |